In [1]:
import numpy as np 
import torch

In [2]:
# in this project we will try to make a character level transformer

In [3]:
# we will start by the embedding matrix , already done before in makemore 

In [4]:
letters = list('qwerty uiopasdfghjklzxcvbnm')

In [5]:
letters.sort()

In [6]:
letters

[' ',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [7]:
## make the map from letters to integers
stoi = {k:i for i,k in enumerate(letters)}
stoi['\n'] = 27

itos = {i:k for k,i in stoi.items() }

In [8]:
stoi

{' ': 0,
 'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '\n': 27}

In [9]:
itos

{0: ' ',
 1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 27: '\n'}

In [10]:
## making the embedding we will start by embedding so first we will make a random data to use

In [11]:
context_size = 10
training_size = 32
vect_dim = 10
X = torch.randint(0,27, size=(training_size, context_size))

In [12]:
X

tensor([[16, 18, 26,  5,  3,  3, 15, 18, 13, 10],
        [17, 15, 12, 12,  8, 12,  4, 14, 24, 23],
        [11, 22, 10, 20, 25, 26,  2, 23, 25, 23],
        [ 3, 10, 19, 14,  6, 12, 13,  1, 16, 26],
        [26,  1, 19, 24,  2,  3,  7, 10,  3,  2],
        [ 9,  5, 18, 16, 11, 25, 17, 21, 25, 24],
        [ 7,  5, 22,  1, 24, 22, 17,  6, 18, 22],
        [13, 19,  7, 11, 26,  3,  3, 16,  5,  0],
        [21,  9, 19,  5, 26, 24,  2, 21,  3, 19],
        [ 0, 25,  7, 11, 25, 10,  0,  2,  3, 19],
        [ 2,  7, 26,  9, 12, 11, 24, 20, 18, 21],
        [ 0, 12, 21,  5, 21, 15,  4, 26,  8,  5],
        [ 3, 15, 19, 14,  2,  4, 25,  2,  2, 14],
        [16, 24, 11, 21, 23, 17,  7, 26,  4,  5],
        [ 9, 14,  2, 26, 19, 15,  8, 17,  0,  7],
        [ 7, 21, 19, 15,  1,  3, 26, 16, 21, 24],
        [18,  7,  7, 19,  9, 13,  5,  4,  2, 10],
        [23,  7, 22,  1, 14, 22, 13,  0,  4, 14],
        [22, 21, 25,  4,  2, 24,  6,  6, 20,  1],
        [18,  9,  9, 11, 10, 15,  7,  5, 15,  4],


In [13]:
## made the data and added zeros before and after for it to represent start and end of the sentence 

In [14]:
import torch.nn as nn

In [15]:
## Here i will put the embedding Class 

In [16]:
class EmbeddingPos(nn.Module):
    def __init__(self, input_size, output_size, context_size): 
        super().__init__()
        self.emb = nn.Embedding(input_size, output_size)
        self.posemb = nn.Embedding(context_size, output_size)
        self.context_size = context_size

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T)
        out = self.emb(x) + self.posemb(pos)
        return out

In [17]:
## so first step is done that ios the embedding matrix , now we go to the next step that is the adding the positionnal information it will be the same as the Embedding class

In [18]:
C = EmbeddingPos(27, 10,context_size)

Xemb = C(X)
Xemb.shape

torch.Size([32, 10, 10])

In [19]:
# now we start with the self attention class

In [20]:
import torch.nn.functional as F
import math

In [21]:
class Self_attention(nn.Module):
    def __init__(self, input_size, output_size): 
        super().__init__()
        self.Wq = nn.Linear(input_size, output_size)
        self.Wk = nn.Linear(input_size, output_size)
        self.Wv = nn.Linear(input_size, output_size)
        self.Wo = nn.Linear(output_size, output_size)
        
    def forward(self, X):
        Q=self.Wq(X) 
        K= self.Wk(X) 
        V= self.Wv(X)
        scores = (Q @ K.transpose(-2, -1))
        # mask the scores before the softmax (replace them with -inf so e(-inf) = 0)
        mask1 = torch.tril(torch.ones(scores.shape[-2:]))
        scores = scores.masked_fill(mask1 == 0, float('-inf'))
    
        attention = F.softmax(scores / math.sqrt(Q.shape[-1]), dim=-1)
        new_vals = attention @ V
        # next is the projection Wo 
        out = self.Wo(new_vals)
       
        return out
    
    @staticmethod
    def AttentionScores(Q, K):
        return Q @ K.transpose(-2, -1)
    @staticmethod
    def AttentionScoresNormalized(Q, K):
        return F.softmax((Q @ K.transpose(-2, -1)) / math.sqrt(Q.shape[-1]), dim=-1)
sf = Self_attention(10, 10)

In [22]:
v = sf(Xemb)
v.shape

torch.Size([32, 10, 10])

In [23]:
v[0] 

tensor([[-0.4195, -0.0086,  0.6517,  0.1771,  0.0738, -0.5818, -0.0842, -0.7953,
          0.3280,  0.4065],
        [ 0.0763, -0.5366,  0.2497, -0.5980,  0.2495,  0.5734, -0.2083, -0.4276,
          1.0652,  0.0797],
        [ 0.0024,  0.1273,  0.7173, -0.0754,  0.4757, -0.5029,  0.2571, -0.3167,
          0.0199,  0.0971],
        [ 0.1396,  0.2310,  0.6549, -0.1130,  0.5045, -0.6098,  0.1932, -0.3184,
          0.0547,  0.1658],
        [ 0.0397,  0.0246,  0.4998, -0.1917,  0.4225, -0.0970,  0.1257, -0.3252,
          0.3397,  0.0422],
        [-0.0663,  0.1389,  0.5210, -0.1054,  0.3804, -0.1989,  0.1583, -0.3804,
          0.2456, -0.0587],
        [ 0.0136,  0.2671,  0.5766, -0.1014,  0.4613, -0.4609,  0.1916, -0.3667,
          0.1002, -0.0194],
        [-0.0022,  0.0824,  0.5258, -0.2038,  0.4074, -0.2744,  0.1016, -0.4076,
          0.2951, -0.0011],
        [ 0.1606,  0.0831,  0.4484, -0.3966,  0.4991, -0.1894,  0.0952, -0.3246,
          0.3566, -0.1232],
        [ 0.0516,  

In [24]:
class FFN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size): 
        ## feed forward network operates at each postion vector unlike the attention 
        super().__init__() 
        self.l1 = nn.Linear(input_size, hidden_size)
        self.gl = nn.GELU()
        self.l2 = nn.Linear(hidden_size, output_size)
        
    def forward(self, X): 
        out = self.l1(X)
        out = self.gl(out) 
        out = self.l2(out)
    
        return out
ffn = FFN(10,40,10)      

In [25]:
ffn(v).shape

torch.Size([32, 10, 10])

In [26]:
class Tranformer(nn.Module):
    def __init__(self, alphabet_size, output_size, context_size):
        super().__init__()
        self.emb = EmbeddingPos(alphabet_size, output_size, context_size)
        self.att = Self_attention(output_size, output_size)
        self.ln1 = nn.LayerNorm(output_size)
        self.ffn = FFN(output_size, 4*output_size, output_size)
        self.ln2 = nn.LayerNorm(output_size)
        self.lm = nn.Linear(output_size, alphabet_size)
    def forward(self,X): 
        Xemb = self.emb(X)
        Xatt = self.att(Xemb) 
        Xatt = Xatt + Xemb
        Xatt = self.ln1(Xatt) 
        Xffn = self.ffn(Xatt)
        Xffn = Xffn+ Xatt
        out = self.ln2(Xffn)
        logits = self.lm(out) ## 27 output to get the real letters back 
        return logits
    
tr = Tranformer(28,10,10)

In [27]:
vt = tr(X)

In [28]:
vt.shape

torch.Size([32, 10, 28])

In [29]:
## now we test and make our first prediction or generation first we make the training 

In [75]:

with open("shakespeare.txt", "r") as file:
    content = file.read()
context_size= 30
encoded_text = [stoi[char] for char in content[:context_size*100000] if char in stoi]
print(len(encoded_text))
X = encoded_text[:context_size*1000]
Y = encoded_text[1:context_size*1000 +1]
X = torch.tensor(X[:context_size*1000]).view(-1, context_size)
Y = torch.tensor(Y[:context_size*1000]).view(-1, context_size)
X.shape, Y.shape

2627599


(torch.Size([1000, 30]), torch.Size([1000, 30]))

In [76]:
from torch.utils.data import TensorDataset, DataLoader

# Model
tr1 = Tranformer(28, 10, context_size)

# Dataset and DataLoader
dataset = TensorDataset(X, Y)

batch_size = 32

train_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True  # Drop the last batch if it contains fewer than 32 samples
)

In [81]:
# Training configuration
num_epochs = 100

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(tr1.parameters(), lr=1e-3)

# Training loop
for epoch in range(num_epochs):
    i = 0
    for batch_X, batch_Y in train_loader:

        # Clear gradients from the previous iteration
        optimizer.zero_grad()

        # Forward pass
        logits = tr1(batch_X)

        # Calculate next-character prediction loss
        loss = criterion(
            logits.permute(0, 2, 1),  # (B, T, V) to (B, V, T)
            batch_Y                   # (B, T)
        )

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()
        if i % 10000 == 0 :
            print(loss)
        i += 1
print(loss) # got 1. at the end somewhat good i guess 

tensor(1.7787, grad_fn=<NllLoss2DBackward0>)
tensor(1.7501, grad_fn=<NllLoss2DBackward0>)
tensor(1.7277, grad_fn=<NllLoss2DBackward0>)
tensor(1.8583, grad_fn=<NllLoss2DBackward0>)
tensor(1.7379, grad_fn=<NllLoss2DBackward0>)
tensor(1.8552, grad_fn=<NllLoss2DBackward0>)
tensor(1.6943, grad_fn=<NllLoss2DBackward0>)
tensor(1.7670, grad_fn=<NllLoss2DBackward0>)
tensor(1.8393, grad_fn=<NllLoss2DBackward0>)
tensor(1.7802, grad_fn=<NllLoss2DBackward0>)
tensor(1.7825, grad_fn=<NllLoss2DBackward0>)
tensor(1.7456, grad_fn=<NllLoss2DBackward0>)
tensor(1.7565, grad_fn=<NllLoss2DBackward0>)
tensor(1.8257, grad_fn=<NllLoss2DBackward0>)
tensor(1.7769, grad_fn=<NllLoss2DBackward0>)
tensor(1.7916, grad_fn=<NllLoss2DBackward0>)
tensor(1.7265, grad_fn=<NllLoss2DBackward0>)
tensor(1.7965, grad_fn=<NllLoss2DBackward0>)
tensor(1.8466, grad_fn=<NllLoss2DBackward0>)
tensor(1.8501, grad_fn=<NllLoss2DBackward0>)
tensor(1.7886, grad_fn=<NllLoss2DBackward0>)
tensor(1.8071, grad_fn=<NllLoss2DBackward0>)
tensor(1.7

In [61]:
prompt = list("to be") 
xt = [stoi[s] for s in prompt]
xt = torch.tensor(xt)
xt = xt.unsqueeze(0)
xt.shape

torch.Size([1, 5])

In [62]:
logits = tr1(xt) 
nxt = logits[:,-1,:] 
nxt = F.softmax(nxt, -1) 


chr_index = nxt[0].argmax().item()
chr_index


0

In [83]:
# Inference
with torch.no_grad():
    prompt = "to be"
    generated_text = prompt
    # Encode prompt
    xt = torch.tensor(
        [stoi[s] for s in prompt],
        dtype=torch.long
    ).unsqueeze(0)  # (1, T)
    for i in range(1000):
        # Keep only the last context_size tokens
        xt_context = xt[:, -context_size:]
        # Forward pass
        logits = tr1(xt_context)
        
        nxt = logits[:, -1, :]  # (1, V)
        probs = F.softmax(nxt, dim=-1)
        chr_index = torch.multinomial(probs[0], num_samples=1).item()
        next_char = itos[chr_index]

        generated_text += next_char
        
        # Add predicted character to input sequence
        next_tensor = torch.tensor(
            [[chr_index]],
            dtype=torch.long,
            device=xt.device
        )
        xt = torch.cat((xt, next_tensor), dim=1)

    print(generated_text)

to be am comm nan
    was ad
   ou
     a the 
    weral my sar it you awes on reks the natull e now whe undevord indam
    oneblet and      no one lor bing tor to ods macke it graniw  womem trow brinbtill tikt
    her whairch he goes a lought dine doral the blelt enelface ss  stractren graciar 
    wous     lis hat for a day at you come to himee nodcere umble e tis shal thin  no bam tefoofn re satuink obeerafs swousere of and bagaat dich loverit  outh
    it cance to nofle be for
  
    o sowteer daunce
    to nva brablot you go fore demw shant mod him mide bad
    and ste of prire vus as will wityse rord odaiktt aretuof rant sats and frake in hey in
     is
    he and bobliol at hope mywee not am dojekgmendd bannecey kighs the  ajerty in     gox cumamd now rear am
    f and is ficrdeil sing hered my bong bre
     sonx at on polaed cankot at blon hoolle their my upor odardaicend forn a foret tor rvee therinallan t in pily
    alarculy that
   
    to kne were thelake his atry man elea